# Diagnóstico de reclutamiento por grupos delictivos — OMSCGR

Notebook de análisis reproducible a partir de la base **EVPSC** (`final.dta`).
Calcula todos los agregados (`APP_DATA`) que alimentan el borrador Word y el
HTML final del producto: conciliación, KPIs, composición del módulo de
exposición/reclutamiento, un modelo de riesgo (regresión logística) y el
desglose territorial (parroquia, zona administrativa, sostenimiento).

**Cómo usar en Google Colab:**
1. Ejecuta la celda de instalación de dependencias.
2. Sube `final.dta` cuando se te pida (o móntalo desde Google Drive).
3. Corre el resto de celdas en orden — cada una imprime sus resultados.
4. Al final se guarda `app_data.json`, descargable desde el panel de archivos.


## 1. Dependencias e insumos

In [ ]:
!pip install statsmodels scikit-learn -q

In [ ]:
# Sube 'final.dta' (recomendado) o móntalo desde Drive si ya lo tienes ahí.
from google.colab import files
import os

if not os.path.exists('final.dta'):
    print("Sube el archivo final.dta:")
    uploaded = files.upload()
    # Si el nombre subido difiere de 'final.dta', renómbralo:
    for fname in uploaded:
        if fname != 'final.dta':
            os.rename(fname, 'final.dta')
print("Archivo listo:", os.path.exists('final.dta'))

In [ ]:
# Alternativa: montar Google Drive en vez de subir el archivo cada vez
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/ruta/a/final.dta'
DATA_PATH = 'final.dta'

In [ ]:
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Cargar la base

`convert_categoricals=False` porque la base ya trae las respuestas como
texto ("1. Sí" / "2. No"), no como categorías numéricas etiquetadas.

In [ ]:
df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Filas:", len(df), " | Columnas:", df.shape[1])
print("Unidades educativas:", df['ue'].nunique())
df.head(3)

## 3. Mapeo de parroquia a administración zonal

Mapeo oficial (Secretaría de Territorio, Hábitat y Vivienda / geoportal
`geoquito.quito.gob.ec`). **Nota**: 4 parroquias periféricas de bajo N
(Mariscal Sucre, San Isidro del Inca, San José de Minas, Nayón) se asignaron
por el criterio geográfico más común y deben validarse si se requiere
precisión administrativa estricta para uso público.

In [ ]:
ZONA_MAP = {
    'CENTRO HISTORICO': 'MANUELA SÁENZ', 'SAN JUAN': 'MANUELA SÁENZ',
    'ITCHIMBIA': 'MANUELA SÁENZ', 'PUENGASI': 'MANUELA SÁENZ',
    'INAQUITO': 'EUGENIO ESPEJO', 'BELISARIO QUEVEDO': 'EUGENIO ESPEJO',
    'KENNEDY': 'EUGENIO ESPEJO', 'COCHAPAMBA': 'EUGENIO ESPEJO',
    'RUMIPAMBA': 'EUGENIO ESPEJO', 'LA CONCEPCION': 'EUGENIO ESPEJO',
    'JIPIJAPA': 'EUGENIO ESPEJO', 'MARISCAL SUCRE': 'EUGENIO ESPEJO',
    'SAN ISIDRO DEL INCA': 'EUGENIO ESPEJO',
    'LA MAGDALENA': 'ELOY ALFARO', 'SAN BARTOLO': 'ELOY ALFARO',
    'CHIMBACALLE': 'ELOY ALFARO', 'CHIMBCALLE': 'ELOY ALFARO',
    'LA FERROVIARIA': 'ELOY ALFARO', 'SOLANDA': 'ELOY ALFARO',
    'CHILIBULO': 'ELOY ALFARO',
    'CHILLOGALLO': 'QUITUMBE', 'QUITUMBE': 'QUITUMBE', 'TURUBAMBA': 'QUITUMBE',
    'GUAMANI': 'QUITUMBE', 'LA ECUATORIANA': 'QUITUMBE',
    'CALDERON (CARAPUNGO)': 'CALDERÓN', 'CALDERON': 'CALDERÓN',
    'POMASQUI': 'LA DELICIA', 'EL CONDADO': 'LA DELICIA', 'COTOCOLLAO': 'LA DELICIA',
    'PONCEANO': 'LA DELICIA', 'COMITE DEL PUEBLO': 'LA DELICIA',
    'SAN JOSE DE MINAS': 'LA DELICIA',
    'TUMBACO': 'TUMBACO', 'CUMBAYA': 'TUMBACO', 'GUAYLLABAMBA': 'TUMBACO',
    'EL QUINCHE': 'TUMBACO', 'CHECA': 'TUMBACO', 'PUEMBO': 'TUMBACO',
    'NAYON': 'TUMBACO',
    'PINTAG': 'LOS CHILLOS', 'CONOCOTO': 'LOS CHILLOS', 'ALANGASI': 'LOS CHILLOS',
    'AMAGUANA': 'LOS CHILLOS',
}

SI, NO = '1. Sí', '2. No'

def pct(series, val=SI):
    """% de 'val' entre respuestas válidas (Sí/No), y n válido."""
    valid = series.isin([SI, NO])
    n = int(valid.sum())
    if n == 0:
        return np.nan, 0
    return 100 * (series[valid] == val).mean(), n

## 4. Conciliación de la base

El módulo de reclutamiento (preguntas PD/EX) se aplicó solo a un subconjunto
de estudiantes — cuestionario dividido en módulos, no un corte por edad.

In [ ]:
mod = df[df['pg8'] != ''].copy()

conciliacion = {
    "n_total": int(len(df)),
    "n_modulo_reclutamiento": int(len(mod)),
    "pct_modulo": round(100 * len(mod) / len(df), 1),
    "n_oferta_valida": int(mod['pg8'].isin([SI, NO]).sum()),
    "n_unidades_educativas": int(df['ue'].nunique()),
}
conciliacion

## 5. KPIs principales

In [ ]:
p_oferta, n_oferta = pct(mod['pg8'])
p_vinc, n_vinc     = pct(mod['ex8_12'])
p_grupos, n_grupos = pct(mod['ex6_1'])
p_h, _ = pct(mod.loc[mod['d1'] == 'Hombre', 'pg8'])
p_m, _ = pct(mod.loc[mod['d1'] == 'Mujer', 'pg8'])

kpis = {
    "oferta_reclutamiento": {"pct": round(p_oferta, 1), "n": n_oferta},
    "conoce_vinculado": {"pct": round(p_vinc, 1), "n": n_vinc},
    "conoce_grupos": {"pct": round(p_grupos, 1), "n": n_grupos},
    "oferta_por_sexo": {"hombre": round(p_h, 1), "mujer": round(p_m, 1)},
}
kpis

## 6. Composición de la exposición (módulo EX)

In [ ]:
items = [
    ('ex1_1', 'Exposición/oferta (ítem 1, 12m)'),
    ('ex6_1', 'Conoce el nombre de grupos delictivos'),
    ('ex8_12', 'Conoce a alguien vinculado a un grupo'),
    ('ex10_14', 'Exposición últimos 12m (ítem 9)'),
    ('ex12_16', 'Exposición últimos 12m (ítem 11)'),
    ('ex18', 'Exposición últimos 12m (ítem 16)'),
    ('ex19', 'Exposición últimos 12m (ítem 17)'),
    ('ex20', 'Exposición últimos 12m (ítem 18)'),
    ('pg8', 'Oferta directa de reclutamiento (12m)'),
]

composicion_exposicion = []
for var, label in items:
    p, n = pct(mod[var])
    composicion_exposicion.append({"var": var, "label": label, "pct": round(p, 1), "n": n})

composicion_exposicion.sort(key=lambda r: r['pct'], reverse=True)
pd.DataFrame(composicion_exposicion)

In [ ]:
# Vista rápida en barras (solo para inspección en el notebook;
# el HTML final usa barras CSS puras, sin librerías de gráficos)
import matplotlib.pyplot as plt

comp_df = pd.DataFrame(composicion_exposicion).sort_values('pct')
plt.figure(figsize=(8, 4))
plt.barh(comp_df['label'], comp_df['pct'], color='#1d9ebe')
plt.xlabel('%')
plt.title('Composición de la exposición al reclutamiento')
plt.tight_layout()
plt.show()

## 7. Modelo de riesgo (regresión logística)

Variable dependiente: `pg8` (oferta directa de reclutamiento en los últimos
12 meses). Variables independientes: sociodemográficas, supervisión
parental, entorno de pares, victimización previa, confianza institucional y
consumo de sustancias.

In [ ]:
d = df.copy()
d['y'] = np.where(d['pg8'] == SI, 1, np.where(d['pg8'] == NO, 0, np.nan))
d['sexo_m'] = np.where(d['d1'] == 'Hombre', 1, np.where(d['d1'] == 'Mujer', 0, np.nan))
d['edad'] = pd.to_numeric(d['d3'], errors='coerce')

map_p17 = {'1. Mucho': 4, '2. Bastante': 3, '3. Poco': 2, '4. Nada': 1}
d['supervision'] = d['p17'].map(map_p17)
d['padres_saben_con_quien'] = np.where(d['p20'] == SI, 1, np.where(d['p20'] == NO, 0, np.nan))
d['familiar_problema'] = np.where(d['p45'] == SI, 1, np.where(d['p45'] == NO, 0, np.nan))

map_p76 = {
    '1. Te harían algún reclamo o te dirían algo para que no lo hicieras': 1,
    '2. Algunos te harían reclamo y otros no': 0.5,
    '3. No te harían ningún reclamo o no te dirían nada': 0,
}
d['amigos_desaprueban'] = d['p76'].map(map_p76)
d['ha_visto_pandillas_barrio'] = np.where(d['p77'] == SI, 1, np.where(d['p77'] == NO, 0, np.nan))

map_p81 = {
    '1. Ninguno': 0, '2. Menos de la mitad': 1, '3. La mitad': 2,
    '4. Más de la mitad': 3, '4. Más de la Mitad': 3,
    '5. Todos o casi todos': 4, '5. Todos o Casi Todos': 4,
}
d['amigos_consumen'] = d['p81'].map(map_p81)

d['victima_robo'] = np.where(d['pi5'] == SI, 1, np.where(d['pi5'] == NO, 0, np.nan))
d['victima_agresion'] = np.where(d['pi6'] == SI, 1, np.where(d['pi6'] == NO, 0, np.nan))
d['victima_amenaza'] = np.where(d['pi9'] == SI, 1, np.where(d['pi9'] == NO, 0, np.nan))
d['confia_policia'] = d['ci2'].map(
    {'1. Desconfío bastante': 1, '2. Desconfío': 2, '3. Confío': 3, '4. Confío bastante': 4})

for v in ['ma1', 'al1', 'ta1', 'co1']:
    d[f'{v}_d'] = np.where(d[v] == SI, 1, np.where(d[v] == NO, 0, np.nan))

cols = ['y', 'sexo_m', 'edad', 'supervision', 'padres_saben_con_quien',
        'familiar_problema', 'amigos_desaprueban', 'ha_visto_pandillas_barrio',
        'amigos_consumen', 'victima_robo', 'victima_agresion', 'victima_amenaza',
        'confia_policia', 'ma1_d', 'al1_d', 'ta1_d', 'co1_d']
sub = d[cols].dropna().copy()
print("N modelo:", len(sub), " | Prevalencia y=1:", round(sub['y'].mean()*100, 1), "%")

In [ ]:
formula = ("y ~ sexo_m + edad + supervision + padres_saben_con_quien + "
           "familiar_problema + amigos_desaprueban + ha_visto_pandillas_barrio + "
           "amigos_consumen + victima_robo + victima_agresion + victima_amenaza + "
           "confia_policia + ma1_d + al1_d + ta1_d + co1_d")

model = smf.logit(formula, data=sub).fit(disp=0)
print(model.summary())

pred = model.predict(sub)
auc = roc_auc_score(sub['y'], pred)
print("\nAUC:", round(auc, 3))

In [ ]:
labels = {
    'sexo_m': 'Ser hombre', 'edad': 'Edad (por año)', 'supervision': 'Supervisión parental',
    'padres_saben_con_quien': 'Padres saben con quién sale',
    'familiar_problema': 'Familiar con problemas',
    'amigos_desaprueban': 'Amigos desaprueban conducta de riesgo',
    'ha_visto_pandillas_barrio': 'Ha visto pandillas en su entorno',
    'amigos_consumen': 'Proporción de amigos que consumen',
    'victima_robo': 'Víctima de robo', 'victima_agresion': 'Víctima de agresión física',
    'victima_amenaza': 'Víctima de amenazas', 'confia_policia': 'Confianza en la Policía',
    'ma1_d': 'Consumo de marihuana', 'al1_d': 'Consumo de alcohol',
    'ta1_d': 'Consumo de tabaco', 'co1_d': 'Consumo de cocaína',
}

factores = []
for var in model.params.index:
    if var == 'Intercept':
        continue
    factores.append({
        "var": var,
        "label": labels.get(var, var),
        "or": round(float(np.exp(model.params[var])), 2),
        "p": round(float(model.pvalues[var]), 4),
        "sig": bool(model.pvalues[var] < 0.05),
    })
factores.sort(key=lambda r: r['or'], reverse=True)

modelo_riesgo = {
    "n": int(len(sub)),
    "auc": round(float(auc), 2),
    "prevalencia_y": round(float(sub['y'].mean()) * 100, 1),
    "factores": factores,
}

or_df = pd.DataFrame(factores)[['label', 'or', 'p', 'sig']]
or_df

## 8. Razones brutas (para el texto narrativo del producto)

In [ ]:
razones_brutas = {}
for var, label in [('pi5', 'robo'), ('pi6', 'agresion'), ('pi9', 'amenaza')]:
    s = mod[mod[var].isin([SI, NO])]
    p_si, _ = pct(s.loc[s[var] == SI, 'pg8'])
    p_no, _ = pct(s.loc[s[var] == NO, 'pg8'])
    razones_brutas[label] = {"pct_si": round(p_si, 1), "pct_no": round(p_no, 1),
                             "razon": round(p_si / p_no, 1)}

for var, label in [('ma1', 'marihuana'), ('co1', 'cocaina'), ('ta1', 'tabaco')]:
    s = mod[mod[var].isin([SI, NO])]
    p_si, _ = pct(s.loc[s[var] == SI, 'pg8'])
    p_no, _ = pct(s.loc[s[var] == NO, 'pg8'])
    razones_brutas[label] = {"pct_si": round(p_si, 1), "pct_no": round(p_no, 1),
                             "razon": round(p_si / p_no, 1)}

pd.DataFrame(razones_brutas).T

## 9. Modelo reducido y estimación fuera de muestra para FISCAL

Las escuelas FISCAL no recibieron el módulo de reclutamiento (`pg8`) **ni**
los bloques de victimización (`pi5/pi6/pi9`) y confianza institucional
(`ci2`) — misma rotación del cuestionario. Reentrenamos un **modelo
reducido**, solo con las variables que sí están disponibles para FISCAL, y
lo aplicamos a esos estudiantes para obtener una probabilidad predicha.

**Esto es una extrapolación fuera de muestra, no una medición.** Asume que
la relación riesgo→reclutamiento estimada en el resto de colegios es
transportable a FISCAL — supuesto que no puede verificarse porque ningún
caso FISCAL tiene la variable de resultado. Esta estimación se usa en la
siguiente sección para completar el desglose territorial.

In [ ]:
covars_reducidas = ['sexo_m', 'edad', 'supervision', 'padres_saben_con_quien',
                    'amigos_desaprueban', 'ha_visto_pandillas_barrio', 'amigos_consumen',
                    'ma1_d', 'al1_d', 'ta1_d', 'co1_d']

train_r = d[['y'] + covars_reducidas].dropna().copy()

formula_reduced = ("y ~ sexo_m + edad + supervision + padres_saben_con_quien + "
                    "amigos_desaprueban + ha_visto_pandillas_barrio + amigos_consumen + "
                    "ma1_d + al1_d + ta1_d + co1_d")
model_r = smf.logit(formula_reduced, data=train_r).fit(disp=0)
auc_r = roc_auc_score(train_r['y'], model_r.predict(train_r))
print("N entrenamiento (modelo reducido):", len(train_r), " | AUC:", round(auc_r, 3))
print("Prevalencia observada en el resto de colegios:", round(train_r['y'].mean() * 100, 1), "%")

# Estudiantes FISCAL: mismas variables, sin el resultado (pg8 nunca se les preguntó)
fiscal_full = d[d['e2'] == 'FISCAL'][['e1', 'e2'] + covars_reducidas].dropna(subset=covars_reducidas).copy()
n_fiscal_total = int((d['e2'] == 'FISCAL').sum())
fiscal_full['prob_pred'] = model_r.predict(fiscal_full)
print("N FISCAL total:", n_fiscal_total, " | con covariables completas:", len(fiscal_full))
print("\nEstimación puntual para FISCAL:", round(fiscal_full['prob_pred'].mean() * 100, 2), "%")

In [ ]:
# Intervalo aproximado por bootstrap (500 iteraciones)
rng = np.random.default_rng(42)
idx = np.arange(len(train_r))
boot = []
for _ in range(500):
    bidx = rng.choice(idx, size=len(idx), replace=True)
    try:
        bm = smf.logit(formula_reduced, data=train_r.iloc[bidx]).fit(disp=0)
        boot.append(bm.predict(fiscal_full[covars_reducidas]).mean())
    except Exception:
        pass
boot = np.array(boot)

estimacion_fiscal = {
    "modelo_reducido": {
        "n_entrenamiento": int(len(train_r)),
        "auc": round(float(auc_r), 2),
        "prevalencia_observada_resto": round(float(train_r['y'].mean()) * 100, 1),
    },
    "n_fiscal_total": n_fiscal_total,
    "n_fiscal_con_covariables": int(len(fiscal_full)),
    "pct_estimado_puntual": round(float(fiscal_full['prob_pred'].mean()) * 100, 2),
    "bootstrap_media": round(float(boot.mean()) * 100, 2),
    "bootstrap_ic95": [round(float(np.percentile(boot, 2.5)) * 100, 2),
                       round(float(np.percentile(boot, 97.5)) * 100, 2)],
}
print(f"Media bootstrap: {estimacion_fiscal['bootstrap_media']}%")
print(f"IC95%: [{estimacion_fiscal['bootstrap_ic95'][0]}, {estimacion_fiscal['bootstrap_ic95'][1]}]")
estimacion_fiscal

In [ ]:
# Perfil de riesgo comparado: FISCAL vs. resto de colegios
labels_perfil = {
    'ma1_d': 'Consumo de marihuana', 'co1_d': 'Consumo de cocaína',
    'ta1_d': 'Consumo de tabaco', 'al1_d': 'Consumo de alcohol',
    'ha_visto_pandillas_barrio': 'Ha visto pandillas en su entorno',
    'amigos_consumen': 'Amigos que consumen (escala 0-4)',
    'amigos_desaprueban': 'Amigos desaprueban conducta de riesgo (escala 0-1)',
    'supervision': 'Supervisión parental (escala 1-4)',
}
perfil_riesgo_comparado = [
    {"var": v, "label": lab, "fiscal": round(float(fiscal_full[v].mean()), 2),
     "resto": round(float(train_r[v].mean()), 2)}
    for v, lab in labels_perfil.items()
]
estimacion_fiscal["perfil_riesgo_comparado"] = perfil_riesgo_comparado
display(pd.DataFrame(perfil_riesgo_comparado))

## 10. Concentración territorial (real + imputado)

Este desglose **combina** la oferta observada (resto de colegios) con la
probabilidad predicha para FISCAL de la sección anterior. Para cada
zona/parroquia/sostenimiento, el % combinado promedia casos reales (0/1) con
casos imputados (probabilidad continua) — es decir, es un valor esperado
sobre la población total, no solo sobre quienes respondieron la pregunta.

In [ ]:
mod['zona'] = mod['e1'].map(ZONA_MAP)
mod['y_real'] = np.where(mod['pg8'] == SI, 1.0, np.where(mod['pg8'] == NO, 0.0, np.nan))
fiscal_full['zona'] = fiscal_full['e1'].map(ZONA_MAP)

def combinar(campo, out_key):
    real = mod[['y_real', campo]].dropna()
    imp = fiscal_full[['prob_pred', campo]].dropna()
    out = []
    for val in sorted(set(real[campo]).union(imp[campo])):
        r = real[real[campo] == val]['y_real']
        f = imp[imp[campo] == val]['prob_pred']
        n_real, n_imp = len(r), len(f)
        n_total = n_real + n_imp
        if n_total == 0:
            continue
        pct_combinado = 100 * (r.sum() + f.sum()) / n_total
        pct_real = 100 * r.mean() if n_real > 0 else None
        out.append({out_key: val, "pct": round(pct_combinado, 1),
                    "pct_solo_real": round(pct_real, 1) if pct_real is not None else None,
                    "n": n_total, "n_real": n_real, "n_imputado": n_imp})
    return out

zonas = sorted(combinar('zona', 'zona'), key=lambda r: r['pct'], reverse=True)
parroquias = sorted([r for r in combinar('e1', 'parroquia') if r['n'] >= 100],
                     key=lambda r: r['pct'], reverse=True)
sostenimiento = sorted(combinar('e2', 'sostenimiento'), key=lambda r: r['pct'], reverse=True)

territorial = {
    "nota": "Combina datos reales (resto de colegios) con datos imputados (FISCAL). "
            "'pct' es el valor combinado; 'pct_solo_real' usa únicamente casos observados.",
    "zonas": zonas, "parroquias": parroquias, "sostenimiento": sostenimiento,
}

print("--- Por zona (combinado) ---")
display(pd.DataFrame(zonas))
print("--- Top parroquias, combinado (n>=100) ---")
display(pd.DataFrame(parroquias).head(10))
print("--- Por sostenimiento, combinado (FISCAL ahora incluido) ---")
display(pd.DataFrame(sostenimiento))

## 11. Guardar `app_data.json`

Este archivo es el insumo único para construir el borrador Word y el HTML
final del producto (`build_html_reclutamiento.py`) — así ninguna cifra se
escribe a mano en ningún paso posterior del pipeline.

In [ ]:
app_data = {
    "conciliacion": conciliacion,
    "kpis": kpis,
    "composicion_exposicion": composicion_exposicion,
    "modelo_riesgo": modelo_riesgo,
    "razones_brutas": razones_brutas,
    "estimacion_fiscal": estimacion_fiscal,
    "territorial": territorial,
}

with open('app_data.json', 'w', encoding='utf-8') as f:
    json.dump(app_data, f, ensure_ascii=False, indent=2)

print("Guardado app_data.json")
print(json.dumps(conciliacion, ensure_ascii=False, indent=2))

**Recomendación**: incorporar el módulo de reclutamiento y victimización a las
escuelas FISCAL en la próxima ronda de la EVPSC. Esta estimación es un
indicio útil para priorizar recursos, no un reemplazo de la medición
directa.

In [ ]:
# Descargar el archivo desde Colab
from google.colab import files
files.download('app_data.json')

---
**Siguiente paso**: usar este `app_data.json` con `build_html_reclutamiento.py`
para regenerar el HTML institucional (línea gráfica OMSCGR / Secretaría de
Seguridad Ciudadana y Gestión de Riesgos), sin volver a tocar la base cruda.